# 14.1 Complexity Analysis

**Prerequisites:** 03 Flow Control (loops), 04 Functions (recursion)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Why we count **operations**, not seconds
- Big-O, and the two siblings nobody mentions (Ω and Θ)
- The growth classes, and what each feels like at scale
- Analysing loops, nested loops and recursion
- **Amortised** analysis - why `list.append` is O(1) despite resizing
- Space complexity, including the stack space recursion uses
- Measuring growth empirically by **doubling n**
- 🔴 When Big-O lies to you
- Interview questions on complexity

---

## Why this notebook comes first

Every other notebook in this folder answers the question *"which structure or algorithm should I use?"* — and the answer is always in terms of complexity. Without this vocabulary the rest is memorisation.

### The problem with timing code

You could just time it. But a measurement in seconds tells you about **your machine, today**: its CPU, its load, your Python version. Run it on a laptop on battery and you get a different answer.

What we actually want to know is: **how does the cost grow as the input grows?** That question has a machine-independent answer.

> **The scaling question.** "It takes 2 seconds" is not useful. "If I give it 10x the data, does it take 10x longer or 100x longer?" is the question that decides whether your system survives next year.

## Counting operations

Take the simplest possible task: find whether a value is in a list.

```
    for item in data:          <- runs once per element
        if item == target:     <- one comparison each time
            return True
    return False
```

For a list of `n` items, the **worst case** is `n` comparisons — the target is last, or absent. Double `n`, double the work. That is **linear**, written **O(n)**.

The cell below counts the comparisons rather than timing them, so the numbers are identical on every machine.

In [ ]:
def linear_search(data, target):
    """Returns (found, comparisons) so we can count the work exactly."""
    comparisons = 0
    for item in data:
        comparisons += 1
        if item == target:
            return True, comparisons
    return False, comparisons


print(f"{'n':>8}{'worst-case comparisons':>26}")
print("-" * 34)
for n in (10, 100, 1_000, 10_000):
    data = list(range(n))
    _found, count = linear_search(data, -1)      # absent: the worst case
    print(f"{n:>8}{count:>26,}")

print("\nEvery 10x in n gives 10x the work. That is O(n) - linear.")
print("No timing involved, so this table is the same on any machine.")

## Big-O, stated properly

> **O(f(n))** describes an **upper bound** on growth: beyond some input size, the cost grows no faster than `f(n)`, ignoring constant factors.

Two rules follow, and they are the ones people apply mechanically:

```
    drop constants        O(2n)      -> O(n)
                          O(n/2)     -> O(n)

    keep only the biggest O(n² + n)  -> O(n²)
                          O(n + 500) -> O(n)
```

**Why dropping constants is legitimate:** we are comparing *shapes of growth*. An O(n) algorithm eventually beats an O(n²) one no matter how much slower each of its steps is — "eventually" is what Big-O is about.

### The three notations

| | Means | Everyday use |
|---|---|---|
| **O(f)** | grows **no faster than** f — upper bound | what everyone says, and usually means the worst case |
| **Ω(f)** | grows **at least as fast as** f — lower bound | "you cannot sort faster than Ω(n log n) by comparisons" |
| **Θ(f)** | both — a **tight** bound | what you often actually mean |

Technically, saying insertion sort is *O(n²)* is true but weak — it is also O(n³), since O is only an upper bound. **Θ(n²)** is the precise claim. In interviews and code review everyone says O and means Θ; know the difference and do not be pedantic about it.

## The growth classes

| Big-O | Name | Example | n = 1,000,000 |
|---|---|---|---|
| **O(1)** | constant | `dict[key]`, `list[i]` | 1 |
| **O(log n)** | logarithmic | binary search (**14.11**) | ~20 |
| **O(n)** | linear | scanning a list | 1,000,000 |
| **O(n log n)** | linearithmic | efficient sorting (**14.10**) | ~20,000,000 |
| **O(n²)** | quadratic | nested loop over the same data | 1,000,000,000,000 |
| **O(2ⁿ)** | exponential | naive recursive subsets | unimaginable |
| **O(n!)** | factorial | all permutations | worse |

### What those numbers mean in practice

At roughly 100 million simple operations per second in Python:

| | n = 1,000 | n = 100,000 | n = 1,000,000 |
|---|---|---|---|
| O(n) | instant | instant | ~0.01 s |
| O(n log n) | instant | ~0.02 s | ~0.2 s |
| O(n²) | ~0.01 s | ~100 s | **~3 hours** |
| O(2ⁿ) | **longer than the universe has existed** | | |

🔴 That O(n²) row is the whole reason this folder exists. A nested loop that is imperceptible on your 1,000-row test data takes **three hours** on a million rows — and nothing in the code looks wrong.

In [ ]:
# Watch the shapes diverge. Operation counts, not timings.
import math

print(f"{'n':>10}{'O(1)':>8}{'O(log n)':>10}{'O(n)':>12}"
      f"{'O(n log n)':>14}{'O(n^2)':>18}")
print("-" * 72)
for n in (10, 100, 1_000, 10_000, 100_000, 1_000_000):
    print(f"{n:>10}{1:>8}{int(math.log2(n)):>10}{n:>12,}"
          f"{int(n * math.log2(n)):>14,}{n * n:>18,}")

print("\nAt n = 1,000,000, O(n log n) is 20 million operations - fine.")
print("O(n^2) is a trillion. Same input, same machine, 50,000x the work.")

## Reading complexity off code

Three rules cover almost everything you will meet.

### 1. Sequential blocks — **add**, then keep the biggest

```
    for x in data: ...        O(n)
    for y in data: ...        O(n)      total O(n + n) = O(2n) = O(n)
```

### 2. Nested loops — **multiply**

```
    for x in data:            n times
        for y in data:        ...n times each
            ...               total O(n x n) = O(n²)
```

🔴 **Nested does not automatically mean n².** What matters is how many times the inner loop *actually* runs:

```
    for i in range(n):
        for j in range(i):    <- runs 0, 1, 2, ... n-1 times
            ...               total n(n-1)/2 = O(n²)   still quadratic

    for i in range(n):
        for j in range(10):   <- always 10
            ...               total O(10n) = O(n)      LINEAR
```

### 3. Halving the problem — **logarithmic**

```
    while n > 1:
        n = n // 2            how many times can you halve n? log₂(n)
```

**The intuition worth keeping:** log₂(1,000,000) ≈ 20. Halving is astonishingly powerful, and it is why sorted data and balanced trees matter so much.

In [ ]:
def count_triangular(n):
    """for i in range(n): for j in range(i) - looks half as bad as n^2."""
    operations = 0
    for i in range(n):
        for _j in range(i):
            operations += 1
    return operations


def count_bounded_inner(n, inner=10):
    """Nested, but the inner loop is a fixed size."""
    operations = 0
    for _i in range(n):
        for _j in range(inner):
            operations += 1
    return operations


def count_halving(n):
    """Each step throws away half the remaining problem."""
    operations = 0
    while n > 1:
        n //= 2
        operations += 1
    return operations


print(f"{'n':>8}{'triangular':>13}{'n^2':>13}{'bounded':>10}{'halving':>10}")
print("-" * 54)
for n in (10, 100, 1_000, 4_000):
    print(f"{n:>8}{count_triangular(n):>13,}{n * n:>13,}"
          f"{count_bounded_inner(n):>10,}{count_halving(n):>10}")

print("\ntriangular is n(n-1)/2 - HALF of n^2, but still O(n^2):")
print("  the constant 1/2 is dropped, and the shape is what matters.")
print("bounded inner loop -> O(n). Nesting alone proves nothing.")
print("halving -> O(log n). 4,000 items in 11 steps.")

## Recursion: complexity from the call tree

For a recursive function, ask two questions:

1. **How many calls happen in total?**
2. **How much work does each call do, excluding its recursive calls?**

Multiply them.

```
    fib(5)                       each call spawns TWO more
    ├── fib(4)                   depth n, branching factor 2
    │   ├── fib(3)               -> roughly 2ⁿ calls
    │   │   ├── fib(2)
    │   │   └── fib(1)
    │   └── fib(2)               <- fib(2) computed AGAIN
    └── fib(3)                   <- and fib(3) AGAIN
```

The duplicated subtrees are the disaster, and fixing them is **dynamic programming** (**14.13**). The cell below counts calls so you can see the explosion, and then shows what a cache does to it.

In [ ]:
import functools

calls = 0


def fib_naive(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


print(f"{'n':>5}{'calls':>14}{'~2^n':>14}")
print("-" * 33)
for n in (5, 10, 20, 25, 30):
    calls = 0
    fib_naive(n)
    print(f"{n:>5}{calls:>14,}{2 ** n:>14,}")

print("\nEach +1 to n roughly DOUBLES the work. n=50 would take days.")

cached_calls = 0


@functools.cache                     # 3.9+; functools.lru_cache before that
def fib_cached(n):
    global cached_calls
    cached_calls += 1
    if n < 2:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)


fib_cached(30)
print(f"\nwith @functools.cache, n=30 took {cached_calls} calls, not "
      f"{2 ** 30:,}")
print("O(2^n) -> O(n), from one decorator. That is 14.13, previewed.")

## 🔴 Amortised analysis - why `list.append` is O(1)

A Python list is a contiguous block of memory. When it fills up, appending must **allocate a bigger block and copy everything across** — which is O(n).

So how can `append` be called O(1)?

Because the list does not grow by one slot; it grows by a **fraction of its current size**. That expensive copy happens rarely, and rarely enough that the cost *averaged over all appends* is constant.

```
    append append append append RESIZE append append ... append RESIZE
      1      1      1      1      n      1      1         1       2n
                                  ^                               ^
                            expensive, but the gap between them doubles
```

> **Amortised O(1)** means: any single call may be expensive, but a sequence of `n` calls costs O(n) in total. That is the guarantee you actually need when appending in a loop.

The cell below watches the underlying capacity grow, using `sys.getsizeof`.

In [ ]:
import sys

data = []
previous_size = sys.getsizeof(data)
resizes = []

for i in range(1, 130):
    data.append(i)
    size = sys.getsizeof(data)
    if size != previous_size:
        resizes.append((i, size))
        previous_size = size

print("the list reallocated after these appends:")
print("   ", [count for count, _ in resizes])
print(f"\n{len(resizes)} reallocations across 129 appends")

gaps = [resizes[i][0] - resizes[i - 1][0] for i in range(1, len(resizes))]
print("gaps between them:", gaps)
print("  ^ the gaps GROW - so the expensive step gets rarer as the list gets")
print("    bigger. Total cost across n appends is O(n): amortised O(1) each.")

print("\n🔴 Contrast: inserting at the FRONT must shift every element, every")
print("   time. No amortisation can save it - see 14.2.")

## Space complexity

The same notation, applied to **memory** instead of time. Count the *extra* space your algorithm needs, beyond the input.

| Pattern | Space |
|---|---|
| a few variables | O(1) — often called *in-place* |
| a copy of the input | O(n) |
| a 2-D table over the input | O(n²) |
| **recursion depth d** | **O(d)** — the call stack is memory too |

🔴 **Recursion costs space that is easy to forget.** A recursive function `n` levels deep holds `n` stack frames. That is why deep recursion raises `RecursionError` — Python's default limit is around 1000 frames, deliberately low to turn a crash into an exception.

> **The classic trade.** You can very often buy time with space: a hash map turns an O(n²) nested scan into an O(n) pass (**14.6**), at the cost of O(n) memory. Recognising that trade is most of practical algorithm design.

In [ ]:
import sys

print("recursion limit:", sys.getrecursionlimit())


def depth_probe(n):
    if n == 0:
        return 0
    return 1 + depth_probe(n - 1)


for n in (100, 900, 5_000):
    try:
        print(f"  depth {n:>5}: reached {depth_probe(n)}")
    except RecursionError:
        print(f"  depth {n:>5}: RecursionError - the stack is finite")

print("\nThe same logic written as a loop uses O(1) stack space:")


def depth_iterative(n):
    total = 0
    while n:
        total += 1
        n -= 1
    return total


print("  iterative to 5,000,000:", depth_iterative(5_000_000))
print("\n  Converting recursion to iteration is a SPACE optimisation.")
print("  Python has no tail-call elimination, so this is on you (14.12).")

## Measuring it: the doubling experiment

Theory tells you the shape. Measurement confirms you implemented that shape.

**The technique:** double `n` and look at the **ratio** of the times.

| If doubling n multiplies time by | The complexity is |
|---|---|
| ~1 | O(1) |
| slightly more than 1 | O(log n) |
| ~2 | O(n) |
| a bit over 2 | O(n log n) |
| ~4 | O(n²) |
| ~8 | O(n³) |

This is the single most useful practical skill in this notebook: it needs no instrumentation, works on code you did not write, and catches an accidental O(n²) immediately.

### Do it two ways

**Count operations** when you can instrument the code: the ratios come out exact, and they are identical on every machine.

**Time it** when you cannot — someone else's function, a library call. This works on anything, and it is **noisy**: background load, cache effects and allocation all interfere. You are looking for *which row of the table it matches*, never a precise number.

> 🔴 The timing pass below is genuinely unreliable on a busy machine. That is not a flaw in the technique — it is what measurement is like, and it is why you reason with counts and confirm with the clock rather than the other way round.

In [ ]:
import time


# ---------------------------------------------------------------
# Part 1: count operations. Exact, and the same on every machine.
# ---------------------------------------------------------------
def linear_ops(n):
    operations = 0
    for _x in range(n):
        operations += 1
    return operations


def quadratic_ops(n):
    operations = 0
    for _x in range(n):
        for _y in range(n):
            operations += 1
    return operations


def logarithmic_ops(n):
    operations = 0
    while n > 1:
        n //= 2
        operations += 1
    return operations


print("counting operations while doubling n:\n")
print(f"{'n':>8}{'O(log n)':>10}{'ratio':>8}{'O(n)':>12}{'ratio':>8}"
      f"{'O(n^2)':>14}{'ratio':>8}")
print("-" * 68)

previous = {}
for n in (250, 500, 1_000, 2_000):
    counts = {
        "log": logarithmic_ops(n),
        "lin": linear_ops(n),
        "quad": quadratic_ops(n),
    }
    parts = [f"{n:>8}"]
    for key, width in (("log", 10), ("lin", 12), ("quad", 14)):
        parts.append(f"{counts[key]:>{width},}")
        if key in previous:
            parts.append(f"{counts[key] / previous[key]:>7.2f}x")
        else:
            parts.append(f"{'-':>8}")
    print("".join(parts))
    previous = counts

print("\n  exactly 1.x, exactly 2.00x, exactly 4.00x - no noise at all,")
print("  because nothing here depends on the machine.")


# ---------------------------------------------------------------
# Part 2: the same shapes on the clock. Noisier, but needs no
# instrumentation - so it works on code you did not write.
# ---------------------------------------------------------------
def measure(func, n, repeats=5):
    """Best-of-N: interference can only ever make a run slower."""
    best = float("inf")
    for _ in range(repeats):
        started = time.perf_counter()
        func(n)
        best = min(best, time.perf_counter() - started)
    return best


print("\n\ntiming the same two functions:\n")
for label, func, sizes, theory in (("O(n)", linear_ops, (250_000, 500_000, 1_000_000), 2),
                                  ("O(n^2)", quadratic_ops, (750, 1_500, 3_000), 4)):
    print(f"  -- {label} --")
    prior = None
    for n in sizes:
        elapsed = measure(func, n)
        if prior is None:
            print(f"     n={n:>9,}  {elapsed * 1000:8.1f} ms")
        else:
            print(f"     n={n:>9,}  {elapsed * 1000:8.1f} ms   "
                  f"measured {elapsed / prior:4.2f}x   theory {theory}x")
        prior = elapsed

print("\n  The measured ratios wander - they depend on what else the machine")
print("  is doing. Run this cell twice and you will get different numbers.")
print("  What does NOT change is which row they point at: one is near 2,")
print("  the other is several times larger. That is the signal.")
print()
print("🔴 An aside worth knowing: an earlier draft used `if x not in seen`")
print("   against a growing list. Its ratios came out at 4.6x then 6.6x -")
print("   worse than quadratic - because the list stopped fitting in cache.")
print("   Big-O cannot see that, which is point 4 in the next section.")

## 🔴 When Big-O lies to you

Big-O is about behaviour as n grows without bound. Real inputs are finite, so it can mislead in five specific ways.

**1. Constants can dominate at real sizes.** An O(n) algorithm with a huge constant loses to an O(n log n) one with a tiny constant, right up to some crossover point you may never reach. This is why `sorted()` uses insertion sort on small runs.

**2. It hides which operation you are counting.** "O(n) comparisons" and "O(n) disk reads" differ by a factor of a million.

**3. Average and worst case can differ enormously.** Quicksort is O(n log n) on average and O(n²) in the worst case. Hash lookup is O(1) average, O(n) worst.

**4. Memory locality is invisible to it.** Two O(n) algorithms can differ 10x because one walks memory sequentially and the other jumps around — cache misses do not appear in the notation.

**5. n may be small and staying small.** For n = 20, an O(n²) solution you can read beats an O(n log n) one you cannot.

> **How to hold both ideas:** use Big-O to *rule out* approaches that cannot possibly scale, then **measure** to choose among the ones that can. It is a filter, not a verdict.

In [ ]:
# Same O(n), 10x apart - because the constant per element differs.
import time

N = 300_000
data = list(range(N))

started = time.perf_counter()
result_a = sum(data)
time_a = time.perf_counter() - started

started = time.perf_counter()
result_b = 0
for x in data:
    result_b += x
time_b = time.perf_counter() - started

started = time.perf_counter()
result_c = 0
for x in data:
    result_c += int(str(x)[0]) * 0 + x        # pointless per-element work
time_c = time.perf_counter() - started

print(f"sum(data)            {time_a * 1000:8.2f} ms   (C loop)")
print(f"python for-loop      {time_b * 1000:8.2f} ms   {time_b / time_a:5.1f}x slower")
print(f"loop + extra work    {time_c * 1000:8.2f} ms   {time_c / time_a:5.1f}x slower")
print(f"\nall three are O(n), all give {result_a == result_b == result_c}")
print("\nBig-O says they are equivalent. The clock disagrees. Both are right:")
print("they scale identically, and the constant factor differs enormously.")

## Interview questions

Complexity comes up in every technical interview, usually as a follow-up to a solution you have just written. Answer in this shape: **state it, justify it, then say whether it can be improved.**

**1. What is the time complexity of your solution?**
> Name time *and* space. "O(n) time and O(n) space, because I build a set of the values in one pass." Volunteering space before being asked reads well.

**2. Why is `list.append` O(1) if the list has to resize?**
> Amortised analysis: the list over-allocates and grows by a fraction of its size, so reallocations become geometrically rarer. Any single append may be O(n); n appends cost O(n) in total.

**3. What is the difference between O, Ω and Θ?**
> Upper bound, lower bound, and both together. Insertion sort is Θ(n²) worst case but Θ(n) on already-sorted input — which is why the *case* must be stated.

**4. Is O(n) always faster than O(n²)?**
> No — only for sufficiently large n. Constants matter below the crossover, which is why real sorts switch to insertion sort for small runs.

**5. What is the complexity of `"x" in my_list` versus `"x" in my_set`?**
> O(n) versus O(1) average. This single distinction is behind an enormous number of accidental O(n²) loops (**14.2**, **14.6**).

**6. Can you do better than O(n log n) for sorting?**
> Not with comparisons — there is an Ω(n log n) lower bound. You can with counting or radix sort, which do not compare, given constraints on the values (**14.10**).

**7. You have an O(n²) solution. How would you find out whether it matters?**
> Ask for the expected size of n and the latency budget, then measure. n=100 makes it irrelevant; n=1,000,000 makes it fatal.

**8. What is the space complexity of a recursive function?**
> At least O(depth) for the call stack, plus whatever each frame holds. A recursion n deep is O(n) space even if it allocates nothing.

---

## Common Mistakes & Pitfalls

1. 🔴 **Assuming nested loops mean O(n²).** What matters is how many times the inner loop runs. A fixed-size inner loop is O(n).
2. 🔴 **Forgetting that `in` on a list is O(n).** Putting it inside a loop is the most common way to write accidental O(n²).
3. 🔴 **Ignoring the space cost of recursion.** Depth n means n stack frames.
4. **Quoting complexity without saying which case.** Quicksort is O(n log n) average and O(n²) worst; the distinction is the whole question.
5. **Treating Big-O as a performance measurement.** It predicts *scaling*, not speed. Two O(n) algorithms can be 10x apart.
6. **Optimising complexity when n is small and fixed.** For n=20, readability wins.
7. **Adding when you should multiply.** Sequential loops add; nested loops multiply.
8. **Forgetting the cost of built-in calls.** `list.insert(0, x)`, `del list[0]` and string concatenation in a loop are all hidden O(n) (**14.2**).

## Best Practices

- State time *and* space complexity whenever you describe an algorithm.
- Use the doubling experiment to verify what you believe you wrote.
- Know the complexity of the built-in operations you use most (**14.2**).
- Prefer a clear O(n log n) over a clever O(n) you cannot explain.
- Say which case you mean - best, average, or worst.
- Trade space for time deliberately, and say so in a comment.
- Profile before optimising; the bottleneck is often not where you assume.
- Write the correct simple version first, then improve it if measurement says so.

## Practice Exercises

Try these before moving on.

1. Determine the complexity of a function with two sequential loops over n, then one with a loop over n containing a loop over m. Why is the second O(n·m) and not O(n²)?
2. Use the doubling experiment on `sorted()` for n = 100k, 200k, 400k. Which ratio do you get, and does it match O(n log n)?
3. 🔴 Write a function that builds a string by `result += piece` in a loop over n pieces. Measure it by doubling. Why is it O(n²), and what does `str.join` change?
4. Instrument `count_triangular` to return the exact count, then confirm algebraically that it equals n(n−1)/2 for n = 10, 100 and 1000.
5. Find the crossover point on your machine between a Python `for` loop and `sum()` — at what n does the difference become noticeable to a user?
6. Take `fib_naive(n)` and count how many times `fib(2)` alone is computed for n = 20. What does that number tell you about what memoisation removes?
7. Given an algorithm that halves the input but does O(n) work at each level, work out the total complexity. (This is the shape of merge sort — **14.10**.)